# Exercises: Short Python Exercises on k-NN, K-Means and Linear Regression

**Practical Machine Learning** · Sayan CHAKI · LIRIS, Université Lyon 2

Twelve short exercises, one for each pen-and-paper exercise of the sheet `Exercises_KNN_KMeans_LinReg.pdf`, on the **same small datasets**. Here you let NumPy do the arithmetic, then check it against scikit-learn. Each exercise should take 10 to 20 minutes.

| Part | Exercises | Topic |
|---|---|---|
| I | 1 to 4 | k-nearest neighbours |
| II | 5 to 7 | K-means |
| III | 8 to 12 | linear regression |

If you already solved the paper sheet, compare your hand results with what the code prints.

**Open in Colab.** In [colab.research.google.com](https://colab.research.google.com) choose *File → Upload notebook* (or open it from Google Drive). Everything uses datasets shipped with scikit-learn, so no download or upload of data is needed.

**Rules of the game**
* Keep all `random_state` / seeds as given, so results are comparable across the class.
* Never use the test set to choose a model or a hyperparameter; it is opened **once**, at the end.
* Cells marked `# TODO` are yours. Cells marked `# CHECK` test your work: run them, they must pass.
* Questions marked ✍️ need a short written answer (2 to 4 sentences, with numbers from your results).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.linear_model import LinearRegression, Ridge, HuberRegressor
from sklearn.metrics import r2_score

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

---
# Part I · k-Nearest Neighbours

## Exercise 1 · Classifying a point
Seven points, two classes (0 = circle, 1 = triangle), and a query $q=(3,2)$.

1. Write `euclidean(P, q)` and `manhattan(P, q)`, each returning the vector of distances from every row of `P` to `q`.
2. Write `knn_vote(P, labels, q, k, dist, weighted=False)`: find the `k` nearest points with `np.argsort(..., kind="stable")` and return the winning class. With `weighted=True`, each neighbour votes with weight `1 / distance`.
3. Print the prediction for `k = 1, 3, 5, 7` (Euclidean), for `k = 3` (Manhattan) and for `k = 5` (weighted, Euclidean), together with the weighted vote totals.

In [ ]:
P = np.array([[1, 3], [2, 2], [3, 4], [4, 1], [5, 3], [2, 5], [6, 2]], dtype=float)
labels = np.array([0, 0, 1, 1, 1, 0, 0])
q = np.array([3.0, 2.0])

In [ ]:
# TODO
def euclidean(P, q):
    raise NotImplementedError("TODO")

def manhattan(P, q):
    raise NotImplementedError("TODO")

def knn_vote(P, labels, q, k, dist, weighted=False):
    raise NotImplementedError("TODO")

In [ ]:
# CHECK (run this cell, it must pass)
assert np.allclose(euclidean(P, q) ** 2, [5, 1, 4, 2, 5, 10, 9])
for k in (1, 3, 5, 7):
    ref = KNeighborsClassifier(n_neighbors=k, algorithm="brute").fit(P, labels).predict([q])[0]
    assert knn_vote(P, labels, q, k, euclidean) == ref, f"k={k}"
ref = KNeighborsClassifier(3, metric="manhattan", algorithm="brute").fit(P, labels).predict([q])[0]
assert knn_vote(P, labels, q, 3, manhattan) == ref
ref = KNeighborsClassifier(5, weights="distance", algorithm="brute").fit(P, labels).predict([q])[0]
assert knn_vote(P, labels, q, 5, euclidean, weighted=True) == ref
print("✅ exercise 1")

### ✍️ Question 1
The prediction changes twice as `k` goes from 1 to 7. Explain each change by pointing at the points that enter the neighbourhood. Why is the `k = 7` answer the same for every possible query?

**Your answer:**

*(2 to 4 sentences, quote numbers from your results)*

## Exercise 2 · Choosing k by leave-one-out
Write `loo_error(x, y, k)` for 1-D data: for each `i`, remove point `i`, predict its label with `k`-NN on the remaining points (majority vote), and count the mistakes. Return the error rate. Compute it for `k = 1, 3, 5, 7` and plot it.

In [ ]:
x = np.array([1, 2, 4, 5, 8, 9, 11, 15], dtype=float)
y = np.array(["A", "A", "A", "B", "B", "A", "B", "B"])

In [ ]:
# TODO
def loo_error(x, y, k):
    raise NotImplementedError("TODO")

In [ ]:
# CHECK (run this cell, it must pass)
for k in (1, 3, 5, 7):
    ref = 1 - cross_val_score(KNeighborsClassifier(k, algorithm="brute"), x[:, None], y, cv=LeaveOneOut()).mean()
    assert np.isclose(loo_error(x, y, k), ref), f"k={k}"
print("✅ exercise 2")

### ✍️ Question 2
Which `k` do you choose? The LOO error at `k = 7` is 100 %: explain this surprising value.

**Your answer:**

*(2 to 4 sentences, quote numbers from your results)*

## Exercise 3 · Why features must be scaled
Customers are described by (age in years, income in euros). Compute the distances from `q` to `a` and to `b`
1. in raw units,
2. after dividing each feature by its training standard deviation `s`,
3. with income in thousands of euros.

Store the three pairs of distances in `raw`, `scaled`, `thousands` (each a length-2 array: distance to `a`, distance to `b`).

In [ ]:
q_c = np.array([30, 40_000.0]); a_c = np.array([32, 52_000.0]); b_c = np.array([55, 41_000.0])
s = np.array([10, 10_000.0])

In [ ]:
# TODO
raw = scaled = thousands = None

In [ ]:
# CHECK (run this cell, it must pass)
assert raw.argmin() == 1 and scaled.argmin() == 0 and thousands.argmin() == 0
assert np.isclose(scaled[0], np.sqrt(0.2**2 + 1.2**2))
print("✅ exercise 3")

### ✍️ Question 3
In the raw computation, by how much does the 25-year age gap of customer `b` change its distance? What general rule for distance-based methods follows?

**Your answer:**

*(2 to 4 sentences, quote numbers from your results)*

## Exercise 4 · The curse of dimensionality
1. Write `edge(p, d)` returning the side of a sub-cube of volume `p` in $[0,1]^d$.
2. **Monte-Carlo check.** Draw `N = 200_000` points uniformly in $[0,1]^d$ (`rng = np.random.default_rng(0)`) and count the fraction that fall inside the cube $[0.5-e/2,\ 0.5+e/2]^d$ with `e = edge(0.01, d)`. It should be close to 1 %. Store the fractions for `d = 1, 2, 10, 50` in `fracs`.

In [ ]:
# TODO
def edge(p, d):
    raise NotImplementedError("TODO")

fracs = []

In [ ]:
# CHECK (run this cell, it must pass)
assert np.isclose(edge(0.01, 2), 0.1)
assert np.allclose(fracs, 0.01, atol=0.002)
print("✅ exercise 4")

---
# Part II · K-Means

## Exercise 5 · Lloyd's algorithm in 1-D
Write `lloyd(x, centres)` for 1-D data: alternate *assign* (nearest centre) and *update* (mean of the assigned points) until the assignments no longer change. Print the centres after each iteration and return the final centres, labels and inertia. Run it from initialisation A $(1, 2)$ and B $(1, 25)$.

In [ ]:
x5 = np.array([1, 2, 3, 8, 9, 10, 25], dtype=float)

In [ ]:
# TODO
def lloyd(x, centres, verbose=True):
    raise NotImplementedError("TODO")

In [ ]:
# CHECK (run this cell, it must pass)
for init in ([1, 2], [1, 25]):
    c, lab, J = lloyd(x5, init, verbose=False)
    km = KMeans(2, init=np.array(init, float)[:, None], n_init=1).fit(x5[:, None])
    assert np.isclose(J, km.inertia_) and np.allclose(np.sort(c), np.sort(km.cluster_centers_.ravel()))
print("✅ exercise 5")

## Exercise 6 · k-means++
The first centre is `1`. Compute the vector `probs` of probabilities that each point is chosen as the second centre. Then simulate 20 000 draws with `rng.choice(x5, p=probs)` (`rng = np.random.default_rng(0)`) and store the empirical frequency of 25 in `freq25`.

In [ ]:
# TODO
probs = None
freq25 = None

In [ ]:
# CHECK (run this cell, it must pass)
assert np.isclose(probs.sum(), 1) and probs[0] == 0
assert abs(freq25 - probs[-1]) < 0.01
print("✅ exercise 6")

## Exercise 7 · The inertia never increases
1. On `Xb, _ = make_blobs(300, centers=4, random_state=1)`, run Lloyd's algorithm in 2-D from 4 random data points (`rng = np.random.default_rng(3)`) for 10 iterations, recording the inertia **after every half-step** (after each assignment and after each update) in a list `J_trace`. Plot it.
2. For one cluster, check numerically that the mean minimises $f(\mu)=\sum_j\lVert x_j-\mu\rVert^2$: evaluate $f$ at the mean and at 1000 random points near it, and store in `mean_is_min` whether no random point does better.

In [ ]:
Xb, _ = make_blobs(300, centers=4, random_state=1)

In [ ]:
# TODO
J_trace = []
mean_is_min = None

In [ ]:
# CHECK (run this cell, it must pass)
assert len(J_trace) == 20
assert all(np.diff(J_trace) <= 1e-9), "the inertia went up at some half-step"
assert mean_is_min
print("✅ exercise 7")

---
# Part III · Linear Regression

All exercises of this part use the five points below.

In [ ]:
x = np.array([1, 2, 3, 4, 5], dtype=float)
y = np.array([2, 3, 5, 4, 6], dtype=float)

## Exercise 8 · The least-squares line
Compute with NumPy (no fitting function): `xbar, ybar, Sxx, Sxy, w, b`, the residuals `e` and `R2`.

In [ ]:
# TODO
xbar = ybar = Sxx = Sxy = w = b = None
e = R2 = None

In [ ]:
# CHECK (run this cell, it must pass)
w_ref, b_ref = np.polyfit(x, y, 1)
assert np.isclose(w, w_ref) and np.isclose(b, b_ref)
assert abs(e.sum()) < 1e-10
assert np.isclose(R2, r2_score(y, w * x + b))
print("✅ exercise 8")

## Exercise 9 · The normal equations
1. Build the design matrix `X` (column of ones, then `x`), compute `XtX`, `Xty` and solve for `theta` with `np.linalg.solve`.
2. Add a third column `2 * x` to get `X3`. Print its rank (`np.linalg.matrix_rank`) and the determinant of `X3.T @ X3`.
3. Solve the Ridge system $(X_3^\top X_3+\lambda I)\theta=X_3^\top y$ with $\lambda=0.1$ → `theta_ridge3`, and print the fitted values.

In [ ]:
# TODO
X = XtX = Xty = theta = None
X3 = theta_ridge3 = None

In [ ]:
# CHECK (run this cell, it must pass)
assert np.allclose(theta, np.linalg.lstsq(X, y, rcond=None)[0])
assert np.linalg.matrix_rank(X3) == 2
assert theta_ridge3.shape == (3,)
print("✅ exercise 9")

### ✍️ Question 4
In the ridge solution, how do the coefficients of `x` and `2x` compare, and why is that the split Ridge prefers? Are the fitted values close to those of the ordinary line?

**Your answer:**

*(2 to 4 sentences, quote numbers from your results)*

## Exercise 10 · Gradient descent
1. Write `grad(w, b)` returning $\left(\partial L/\partial w,\ \partial L/\partial b\right)$ for $L=\frac1n\sum(y_i-wx_i-b)^2$, and check it against finite differences $\frac{L(w+h,b)-L(w-h,b)}{2h}$.
2. Perform one step from $(0,0)$ with $\eta=0.05$ → `w1, b1`, and the loss after it → `L1`.
3. Run 200 steps from $(0,0)$ with $\eta=0.05$ on `x`, then on the **centred** feature `x - x.mean()` (predicting with $w(x-\bar x)+b$). Plot both loss curves on a log scale (subtract the optimal loss).

In [ ]:
# TODO
def loss(w, b, xx=x):
    return np.mean((y - w * xx - b) ** 2)

def grad(w, b, xx=x):
    raise NotImplementedError("TODO")

w1 = b1 = L1 = None

In [ ]:
# CHECK (run this cell, it must pass)
h = 1e-6
for (w0, b0) in [(0.0, 0.0), (0.7, 2.0)]:
    gw, gb = grad(w0, b0)
    assert np.isclose(gw, (loss(w0 + h, b0) - loss(w0 - h, b0)) / (2 * h), atol=1e-5)
    assert np.isclose(gb, (loss(w0, b0 + h) - loss(w0, b0 - h)) / (2 * h), atol=1e-5)
assert np.isclose(L1, loss(w1, b1)) and L1 < loss(0, 0)
print("✅ exercise 10")

### ✍️ Question 5
Why does centring `x` speed up gradient descent so much? (With centred `x`, what is the optimal intercept, and do the two gradients still depend on each other?)

**Your answer:**

*(2 to 4 sentences, quote numbers from your results)*

## Exercise 11 · Ridge in one dimension
Write `ridge_slope(lam)` using the closed form $w_\lambda=S_{xy}/(S_{xx}+\lambda)$ and plot $w_\lambda$ for $\lambda\in[0,100]$. Store `ridge_slope(5)` in `w5` and the matching intercept in `b5`.

In [ ]:
# TODO
def ridge_slope(lam):
    raise NotImplementedError("TODO")

w5 = b5 = None

In [ ]:
# CHECK (run this cell, it must pass)
for lam in (0, 5, 10, 90):
    m = Ridge(alpha=lam if lam > 0 else 1e-12).fit(x[:, None], y)
    assert np.isclose(ridge_slope(lam), m.coef_[0])
assert np.isclose(b5, Ridge(alpha=5).fit(x[:, None], y).intercept_)
print("✅ exercise 11 (scikit-learn's Ridge does not penalise the intercept either)")

## Exercise 12 · Outliers and leverage
1. Refit the least-squares line after adding $(3,14)$, then after adding $(9,4)$ (separately). Store the slopes in `slope_a`, `slope_b`.
2. For the second dataset, compute the **leverages** $h_{ii}$, the diagonal of $H=X(X^\top X)^{-1}X^\top$ → `h`. Which point has the largest leverage?
3. Fit `HuberRegressor()` on the second dataset and compare its slope with least squares.

In [ ]:
# TODO
slope_a = slope_b = None
h = None

In [ ]:
# CHECK (run this cell, it must pass)
assert np.isclose(slope_a, 0.9) and np.isclose(slope_b, 0.225)
assert np.isclose(h.sum(), 2), "the leverages must sum to the number of parameters"
assert h.argmax() == 5
print("✅ exercise 12")

### ✍️ Question 6
Relate the leverage of the point $(9,4)$ to its effect on the slope. Why do the leverages sum to 2? Did the Huber loss help here, and why or why not?

**Your answer:**

*(2 to 4 sentences, quote numbers from your results)*